<a href="https://colab.research.google.com/github/SiriusDarkz/riesgo-mora-cooperativa/blob/main/caso1_cooperativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Caso 1 · Riesgo de mora en préstamos nuevos
## Cooperativa Progreso del Sur

**Asignatura:** Selección y Validación de Modelos  
**Profesor:** Dr. Edian Franco  
**Equipo:** Jose Eugenio Duran Vizcaino · Anthony Burgos · Isaac Sanchez ·
Maximo Martinez · Francisco Jose Mejia

---

## El caso

La **Cooperativa Progreso del Sur** ofrece préstamos personales, comerciales
y para mejoras de vivienda a través de sucursales en Santo Domingo, San
Cristóbal, Baní y Azua. Durante el último año aumentaron las solicitudes
recibidas por canales digitales, pero el equipo de riesgo continúa revisando
manualmente buena parte de los expedientes.

La gerencia observa que algunas personas presentan atrasos importantes durante
los primeros meses del préstamo. Cuando esto ocurre, la cooperativa debe
realizar llamadas de cobro, renegociar condiciones y aumentar las provisiones
financieras. La gerente de riesgo plantea la necesidad de **identificar cuáles
solicitudes nuevas podrían presentar una mora superior a 30 días durante sus
primeros seis meses**.

Dos restricciones definen el problema:

- **Capacidad limitada:** el equipo de analistas solo puede revisar en
  detalle 120 solicitudes por semana.
- **Sin rechazo automático:** la gerencia no desea rechazar solicitantes con
  el modelo; desea priorizar cuáles expedientes necesitan verificación
  adicional.

## Qué construye este proyecto

> Un procedimiento que, cada semana, ordena las solicitudes nuevas por riesgo
> de mora y selecciona las 120 que el equipo de analistas debe revisar en
> detalle. La revisión funciona como tratamiento preventivo: verifica ingresos,
> pide garantías o ajusta condiciones, y así evita una parte de las moras que
> iban a ocurrir. El modelo no aprueba ni rechaza a nadie, solo apunta la
> capacidad limitada de revisión hacia donde más pérdida puede prevenir. Se
> recomendará implementarlo únicamente si demuestra, en un test honesto, que
> apunta mejor que la regla actual de la cooperativa.

Todo el ejercicio utiliza **datos sintéticos** generados en Python con semilla
fija, siguiendo las 8 etapas del protocolo del curso: formular, generar datos,
auditar variables, diseñar la evaluación, comparar contra baselines, congelar
el protocolo, evaluar en test una sola vez y recomendar.

# Etapa 1 · Formulación del problema

**Actor.** La gerencia de riesgo de la Cooperativa Progreso del Sur: la
gerente de riesgo define la política de revisión y su equipo de analistas la
ejecuta.

**Decisión.** Cuáles solicitudes nuevas de préstamo se envían cada semana a
verificación adicional, dentro del límite operativo de 120 revisiones
semanales.

**Acción.** Revisión manual detallada del expediente, que puede derivar en
verificación de ingresos, solicitud de garantías, reducción del monto o
cambio del plazo. La predicción no rechaza solicitantes: prioriza cuáles
revisar.

**Unidad de análisis.** Una solicitud de préstamo (una fila = una solicitud).
No es el cliente: un mismo cliente puede presentar varias solicitudes, lo que
obliga a controlar que sus solicitudes no queden repartidas entre desarrollo
y test.

**Momento de predicción.** Al recibir la solicitud completa, antes de la
decisión de aprobación. Elegimos este momento (y no "antes del desembolso")
porque la revisión adicional sirve precisamente para informar las condiciones
de aprobación. Consecuencias: (a) la variable `approved` aún no existe al
predecir, por lo que no puede usarse como predictora; (b) solo las
solicitudes aprobadas y desembolsadas llegan a tener etiqueta observada — una
limitación (etiquetas selectivas) que declaramos en el informe final.

**Horizonte.** Los primeros 6 meses de vida del préstamo, contados desde el
desembolso. La etiqueta de un préstamo solo se conoce cuando esta ventana se
cierra. *Supuesto de madurez:* asumimos que el análisis se realiza en una
fecha en la que todos los préstamos simulados ya completaron su ventana de
6 meses; en producción, la cooperativa solo podría entrenar con solicitudes
desembolsadas al menos 6 meses atrás, y las más recientes aún no tendrían
etiqueta observada.

**Variable objetivo.** `default_30d`: vale 1 si el préstamo alcanza una mora
superior a 30 días en algún momento durante sus primeros 6 meses; 0 en caso
contrario. Se deriva del seguimiento de atrasos (`days_past_due_6m`). Nótese
que combina dos números con roles distintos: los 30 días definen la severidad
del atraso que cuenta como evento; los 6 meses definen la ventana de
observación.

**Capacidad operativa.** 120 solicitudes por semana pueden recibir revisión
detallada. Esto convierte el problema en uno de **priorización**: el
procedimiento ordena las solicitudes de cada semana por riesgo estimado y las
120 primeras se revisan.

**Costo de los errores.**
- *Falso negativo* (no revisar una solicitud que luego cae en mora): costo
  financiero directo — provisiones, gestión de cobranza, renegociación y
  posible pérdida de capital.
- *Falso positivo* (gastar una revisión en una solicitud que habría pagado
  bien): horas de analista y fricción para un buen socio. Con capacidad fija,
  cada falso positivo tiene además un costo de oportunidad: desplaza del top
  120 a una solicitud riesgosa.

**Supuestos de costo (ilustrativos, fijados antes de la evaluación).**
Para traducir los errores a magnitudes comparables adoptamos supuestos
redondos, coherentes con los datos sintéticos que generaremos:

| Concepto | Supuesto |
|---|---|
| Monto promedio del préstamo | RD\$150,000 |
| Pérdida esperada si hay mora >30d (provisiones, cobranza, pérdida) | ≈20% del monto → RD\$30,000 |
| Costo de una revisión manual (≈2 horas de analista) | RD\$1,000 |
| Efecto de la revisión sobre una solicitud riesgosa | reduce la probabilidad de mora ≈35% |

Implicación: un falso negativo cuesta ~30 veces más que un falso positivo;
por eso consideramos más costoso el falso negativo y la métrica principal
medirá cuántas moras reales se capturan dentro de la capacidad semanal.
Además, el beneficio esperado de revisar un caso realmente riesgoso
(0.35 × RD\$30,000 ≈ RD\$10,500) supera con holgura el costo de la revisión
(RD\$1,000), lo que valida usar la capacidad completa.

El 35% es un **parámetro de diseño de la simulación**: el enunciado exige que
la revisión pueda disminuir la mora observada; nosotros fijamos su magnitud
en un valor moderado y plausible, y el generador de datos de la Etapa 2 usará
este mismo parámetro. Como el tratamiento es idéntico para cualquier método
de selección, la comparación entre el modelo y la regla vigente no depende
del valor exacto: variaciones razonables (20%–50%) cambian las cifras en
pesos, no la conclusión. Estas cifras son supuestos del ejercicio; en una
implementación real se calibrarían con la contabilidad de la cooperativa y
las normas de provisión aplicables.

**Criterio de no implementación.** El procedimiento no se recomienda si, con
las mismas 120 revisiones semanales en el período de test, no captura más
moras futuras que la regla operativa vigente (priorizar solicitudes con
deuda/ingreso alta y atrasos previos). Tampoco si la mejora, traducida a
dinero con los supuestos de costo anteriores, resulta demasiado pequeña para
compensar el costo de construir, mantener y monitorear el modelo.

# Etapa 2 · Generación de datos sintéticos

Generamos los datos en el orden en que ocurren en la realidad: primero las
personas y sus perfiles, luego el riesgo que ese perfil implica, después la
política de revisión de la cooperativa, la intervención sobre los revisados,
el resultado de los 6 meses y, al final, las secuelas de cobranza. Ese orden
es lo que hace que la fuga y la intervención existan de verdad en los datos:
las variables de fuga se calculan a partir del resultado, y la revisión
reduce la probabilidad de mora antes de sortear el desenlace (el factor 0.65
que fijamos en la Etapa 1).

Decisiones de diseño:

- **Volumen creciente.** El caso cuenta que las solicitudes aumentaron por
  los canales digitales. Lo reflejamos: el volumen semanal crece de ~110 a
  ~240 solicitudes. Al inicio del período, las 120 revisiones semanales
  alcanzaban para casi todo; al final cubren menos de la mitad. Esa es la
  razón de ser del proyecto, contada por los propios datos.
- **Cambio temporal.** Con el canal digital llega un perfil distinto (más
  joven, menos antigüedad laboral, menos historial) y un deterioro gradual
  de la tasa base. El futuro no se parece al pasado, y la partición de la
  Etapa 4 tendrá que respetarlo.
- **Entidades repetidas.** Las solicitudes se sortean sobre un padrón de
  clientes, así que un mismo cliente puede aparecer varias veces con datos
  coherentes (misma persona, edad que avanza).
- **Política histórica con capacidad.** La revisión manual del pasado sigue
  la regla vigente (deuda/ingreso alta y atrasos previos) limitada a 120 por
  semana — exactamente el baseline operativo contra el que competirá el
  modelo.
- **Faltantes con mecanismo.** `months_in_job` viene vacía para empleo
  informal y `payment_history_score` para clientes sin historial.

Todos los parámetros están declarados al inicio del código; la semilla es 42
y el conteo exacto de filas queda fijado por ella (~18,000, dentro del rango
exigido).

In [3]:
# =====================================================================
# Etapa 2 · Generador de datos sintéticos — Caso 1
# Parte 1: parámetros del mundo simulado y padrón de clientes
# =====================================================================
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)

P = {
    # --- calendario y volumen ---
    "fecha_inicio":        "2024-01-01",   # lunes de la semana 1
    "n_semanas":           104,            # 24 meses
    "volumen_inicial":     110,            # solicitudes/semana al inicio
    "volumen_final":       240,            # .../semana al final (crecimiento digital)
    "prop_digital_inicial": 0.20,          # proporción de solicitudes de perfil digital
    "prop_digital_final":   0.60,
    # --- decisiones de la Etapa 1 ---
    "capacidad_semanal":   120,            # revisiones manuales posibles por semana
    "efecto_revision":     0.65,           # la revisión multiplica la prob. de mora (reducción 35%)
    # --- geografía ---
    "sucursales":          ["Santo Domingo", "San Cristóbal", "Baní", "Azua"],
    "riesgo_sucursal":     {"Santo Domingo": 0.00, "San Cristóbal": 0.15,
                            "Baní": -0.10, "Azua": 0.25},   # ajuste en log-odds
}

def generar_clientes(rng, n, cohorte, id_inicio):
    """Crea un padrón de clientes. La cohorte 'digital' es más joven, con
    menos antigüedad y menos historial crediticio que la 'tradicional'."""
    if cohorte == "tradicional":
        edad0    = np.clip(rng.normal(42, 11, n), 22, 75)
        ingreso0 = np.clip(rng.lognormal(np.log(40000), 0.45, n), 12000, 400000)
        empleo   = rng.choice(["formal", "informal", "independiente", "pensionado"],
                              n, p=[0.55, 0.20, 0.17, 0.08])
        p_historial, p_suc = 0.78, [0.38, 0.22, 0.20, 0.20]
        calidad  = rng.normal(0.00, 1.0, n)     # riesgo latente (mayor = peor)
    else:
        edad0    = np.clip(rng.normal(31, 7, n), 20, 60)
        ingreso0 = np.clip(rng.lognormal(np.log(33000), 0.50, n), 12000, 300000)
        empleo   = rng.choice(["formal", "informal", "independiente", "pensionado"],
                              n, p=[0.45, 0.32, 0.21, 0.02])
        p_historial, p_suc = 0.55, [0.55, 0.20, 0.13, 0.12]
        calidad  = rng.normal(0.15, 1.0, n)     # levemente más riesgosa en promedio

    antiguedad0 = np.select(
        [empleo == "formal", empleo == "independiente", empleo == "pensionado"],
        [rng.exponential(48, n), rng.exponential(36, n), rng.exponential(90, n)],
        default=rng.exponential(18, n))        # informal (luego será faltante)

    atrasos = np.minimum(rng.poisson(0.6 * np.exp(0.6 * calidad)), 6)
    score   = np.round(np.clip(68 - 9*calidad - 4*atrasos + rng.normal(0, 6, n), 5, 99))
    score   = np.where(rng.random(n) < p_historial, score, np.nan)  # sin historial → vacío

    return pd.DataFrame({
        "client_id":             [f"C{id_inicio + i:06d}" for i in range(n)],
        "cohorte":               cohorte,
        "branch_id":             rng.choice(P["sucursales"], n, p=p_suc),
        "employment_type":       empleo,
        "prior_late_payments":   atrasos,
        "payment_history_score": score,
        "edad0":                 edad0,        # internas: se eliminan al final
        "ingreso0":              ingreso0,
        "antiguedad0":           antiguedad0,
        "calidad":               calidad,
    })

clientes_trad = generar_clientes(rng, 20000, "tradicional", id_inicio=100000)
clientes_dig  = generar_clientes(rng, 20000, "digital",     id_inicio=300000)
print(f"Padrón creado: {len(clientes_trad):,} tradicionales + {len(clientes_dig):,} digitales")

Padrón creado: 20,000 tradicionales + 20,000 digitales


In [4]:
# =====================================================================
# Parte 2: solicitudes semana a semana, riesgo, política histórica,
# intervención, desenlaces y variables de fuga
# =====================================================================

# --- 2.1 Sortear las solicitudes de cada semana ---------------------
lunes  = pd.date_range(P["fecha_inicio"], periods=P["n_semanas"], freq="7D")
vol    = np.linspace(P["volumen_inicial"], P["volumen_final"], P["n_semanas"])
p_dig  = np.linspace(P["prop_digital_inicial"], P["prop_digital_final"], P["n_semanas"])

partes = []
for w in range(P["n_semanas"]):
    n_w = rng.poisson(vol[w])
    es_digital = rng.random(n_w) < p_dig[w]
    idx_d = rng.integers(0, len(clientes_dig),  int(es_digital.sum()))
    idx_t = rng.integers(0, len(clientes_trad), int((~es_digital).sum()))
    base = pd.concat([clientes_dig.iloc[idx_d], clientes_trad.iloc[idx_t]],
                     ignore_index=True)
    base["semana_idx"] = w
    base["application_date"] = lunes[w] + pd.to_timedelta(
        rng.integers(0, 5, len(base)), unit="D")          # lunes a viernes
    partes.append(base)

df = pd.concat(partes, ignore_index=True)
n = len(df)

# --- 2.2 Variables de la solicitud ----------------------------------
df["age"] = np.round(df["edad0"] + df["semana_idx"] / 52).astype(int)
df["monthly_income"] = np.round(df["ingreso0"] * rng.normal(1, 0.04, n), -2)
df["requested_amount"] = np.round(np.clip(
    df["monthly_income"] * rng.lognormal(np.log(3.6), 0.50, n), 10000, 1500000), -3)
df["loan_term_months"] = rng.choice([6, 12, 18, 24, 36, 48], n,
                                    p=[0.08, 0.22, 0.18, 0.28, 0.16, 0.08])

# deuda total y razón deuda/ingreso ANUAL (así definimos debt_to_income)
dti_lat = np.clip(rng.normal(0.32 + 0.06*df["calidad"]
                             + np.where(df["cohorte"] == "digital", 0.05, 0),
                             0.16), 0.02, 1.40)
df["existing_debt"]  = np.round(dti_lat * df["monthly_income"] * 12, -2)
df["debt_to_income"] = (df["existing_debt"] / (df["monthly_income"] * 12)).round(3)

antig = df["antiguedad0"] + df["semana_idx"] * 0.23          # la antigüedad avanza
antig = np.where(df["employment_type"] == "informal", np.nan, antig)
antig = np.where(rng.random(n) < 0.02, np.nan, antig)        # 2% faltante adicional
df["months_in_job"] = np.round(antig)

# --- 2.3 Riesgo verdadero (antes de cualquier intervención) ---------
dti_c    = np.clip(df["debt_to_income"], 0, 1.2)
score    = df["payment_history_score"]
ef_score = np.where(score.isna(), 0.25, -0.45 * (score - 65) / 25)
tenure   = df["months_in_job"]
ef_ten   = np.where(tenure.isna(), 0.0, -0.012 * np.minimum(tenure, 120))
ef_emp   = df["employment_type"].map({"formal": 0.0, "informal": 0.50,
                                      "independiente": 0.28, "pensionado": -0.15}).to_numpy()
ef_suc   = df["branch_id"].map(P["riesgo_sucursal"]).to_numpy()
ef_edad  = -0.008 * (df["age"] - 38)
ef_tiempo = 0.25 * df["semana_idx"] / (P["n_semanas"] - 1)   # deterioro gradual

z = (-2.55 + 1.35*dti_c + 0.28*np.minimum(df["prior_late_payments"], 5)
     + ef_score + ef_ten + ef_emp + ef_suc + ef_edad + ef_tiempo
     + 0.50*df["calidad"] + rng.normal(0, 0.35, n))          # ruido irreducible
df["p_mora_sin_intervencion"] = (1 / (1 + np.exp(-z))).round(4)

# --- 2.4 Política histórica de revisión (el baseline operativo) -----
# Regla vigente: prioridad por deuda/ingreso y atrasos previos,
# con adherencia imperfecta, limitada a la capacidad semanal.
df["_prioridad"] = (2.0*dti_c + 0.5*np.minimum(df["prior_late_payments"], 3)
                    + rng.normal(0, 0.15, n))
df["_rank"] = df.groupby("semana_idx")["_prioridad"].rank(ascending=False, method="first")
df["manual_review"] = (df["_rank"] <= P["capacidad_semanal"]).astype(int)

# --- 2.5 Intervención: la revisión reduce el riesgo -----------------
p_final = df["p_mora_sin_intervencion"] * np.where(df["manual_review"] == 1,
                                                   P["efecto_revision"], 1.0)

# --- 2.6 Aprobación (rechazo solo en riesgo extremo) ----------------
p_aprob = np.where(df["p_mora_sin_intervencion"] > 0.55, 0.50, 0.965)
df["approved"] = rng.binomial(1, p_aprob)
aprob = df["approved"] == 1

# --- 2.7 Desenlace de los 6 meses (solo préstamos desembolsados) ----
mora = np.zeros(n)
mora[aprob] = rng.binomial(1, p_final[aprob])

dpd = np.full(n, np.nan)
con_mora = aprob & (mora == 1)
sin_mora = aprob & (mora == 0)
dpd[con_mora] = np.minimum(31 + np.round(rng.exponential(28, con_mora.sum())), 180)
leves = rng.random(sin_mora.sum()) < 0.28                    # atrasos leves (1–30 días)
dpd[sin_mora] = np.where(leves, rng.integers(1, 31, sin_mora.sum()), 0)

df["days_past_due_6m"] = dpd
df["default_30d"] = np.where(aprob, mora, np.nan)            # sin desembolso no hay etiqueta

# --- 2.8 Variables de fuga: nacen del resultado ---------------------
lam_llamadas = np.where(mora == 1, 6.0, np.where(dpd > 0, 1.6, 0.15))
llamadas = np.full(n, np.nan)
llamadas[aprob] = rng.poisson(lam_llamadas[aprob])
df["collection_calls_6m"] = llamadas

reestr = np.full(n, np.nan)
reestr[aprob] = np.where(mora[aprob] == 1, rng.binomial(1, 0.35, int(aprob.sum())), 0)
df["restructured_after_default"] = reestr

legal = np.full(n, np.nan, dtype=object)
legal[sin_mora] = "ninguna"
legal[con_mora] = rng.choice(["ninguna", "prejudicial", "judicial"],
                             int(con_mora.sum()), p=[0.35, 0.45, 0.20])
df["legal_collection_status"] = legal

print(f"Solicitudes generadas: {n:,}")

Solicitudes generadas: 18,057


In [5]:
# =====================================================================
# Parte 3: dataset final, archivo CSV y control de calidad
# =====================================================================
COLUMNAS = [
    "application_date", "branch_id", "client_id", "age", "monthly_income",
    "requested_amount", "loan_term_months", "existing_debt", "employment_type",
    "months_in_job", "prior_late_payments", "payment_history_score",
    "debt_to_income", "manual_review", "approved", "days_past_due_6m",
    "default_30d", "collection_calls_6m", "restructured_after_default",
    "legal_collection_status", "p_mora_sin_intervencion",
]
df = (df[COLUMNAS + ["semana_idx"]]
      .sort_values("application_date")
      .reset_index(drop=True))

df[COLUMNAS].to_csv("solicitudes_sinteticas.csv", index=False)

# ---------------- Control de calidad --------------------------------
apro = df["approved"] == 1
rev  = df["manual_review"] == 1
vc   = df["client_id"].value_counts()

print("=== CONTROL DE CALIDAD — requisitos del enunciado ===\n")
print(f"Observaciones: {len(df):,}  (requisito: entre 3,000 y 20,000)")
print(f"Rango temporal: {df.application_date.min().date()} → {df.application_date.max().date()}"
      f"  ({df.semana_idx.nunique()} semanas)")
print(f"Solicitudes de clientes repetidos: {df.client_id.map(vc).ge(2).mean():.1%}")
print(f"Aprobadas y desembolsadas: {apro.mean():.1%}")
print(f"Tasa de mora >30d (entre aprobadas): {df.loc[apro,'default_30d'].mean():.1%}")
print(f"Monto promedio solicitado: RD {df.requested_amount.mean():,.0f}"
      f"  (supuesto Etapa 1: RD 150,000)")
print(f"Faltantes — months_in_job: {df.months_in_job.isna().mean():.1%} | "
      f"payment_history_score: {df.payment_history_score.isna().mean():.1%}")

rev_sem = df.groupby("semana_idx")["manual_review"].sum()
print(f"\nRevisiones por semana: mín {rev_sem.min()}, máx {rev_sem.max()}"
      f"  (capacidad: {P['capacidad_semanal']})")
print(f"Cobertura de la revisión: {rev.mean():.1%} de las solicitudes"
      f"  (al inicio alcanzaba casi todo; al final, menos de la mitad)")

mitad = P["n_semanas"] // 2
mora_1 = df.loc[apro & (df.semana_idx <  mitad), "default_30d"].mean()
mora_2 = df.loc[apro & (df.semana_idx >= mitad), "default_30d"].mean()
print(f"\nCambio temporal — mora año 1: {mora_1:.1%} | año 2: {mora_2:.1%}")

obs  = df.loc[apro & rev, "default_30d"].mean()
sin_ = df.loc[apro & rev, "p_mora_sin_intervencion"].mean()
print(f"Intervención — entre revisadas: mora observada {obs:.1%} "
      f"vs riesgo sin revisión {sin_:.1%}  (la revisión 'borra' la diferencia)")

corr = df.loc[apro, ["collection_calls_6m", "default_30d"]].corr().iloc[0, 1]
print(f"Fuga — correlación llamadas de cobro vs mora: {corr:.2f}  (el eco de la respuesta)")

chequeo = (df.loc[apro, "default_30d"] == (df.loc[apro, "days_past_due_6m"] > 30)).all()
print(f"\nConsistencia target: default_30d == (days_past_due_6m > 30) → {chequeo}")

=== CONTROL DE CALIDAD — requisitos del enunciado ===

Observaciones: 18,057  (requisito: entre 3,000 y 20,000)
Rango temporal: 2024-01-01 → 2025-12-26  (104 semanas)
Solicitudes de clientes repetidos: 37.0%
Aprobadas y desembolsadas: 94.5%
Tasa de mora >30d (entre aprobadas): 12.0%
Monto promedio solicitado: RD 168,038  (supuesto Etapa 1: RD 150,000)
Faltantes — months_in_job: 27.0% | payment_history_score: 31.3%

Revisiones por semana: mín 100, máx 120  (capacidad: 120)
Cobertura de la revisión: 68.5% de las solicitudes  (al inicio alcanzaba casi todo; al final, menos de la mitad)

Cambio temporal — mora año 1: 10.7% | año 2: 12.9%
Intervención — entre revisadas: mora observada 12.8% vs riesgo sin revisión 20.2%  (la revisión 'borra' la diferencia)
Fuga — correlación llamadas de cobro vs mora: 0.82  (el eco de la respuesta)

Consistencia target: default_30d == (days_past_due_6m > 30) → True


# Etapa 3 · Auditoría de variables

Clasificamos cada variable del dataset según el papel que puede cumplir en el
proyecto. El criterio es el mismo para todas, la pregunta del enunciado:

> ¿Esta variable existiría, con el mismo valor y significado, al momento de
> decidir?

El momento de decidir quedó definido en la Etapa 1: cuando la solicitud llega
completa, antes de aprobarla. Hicimos esta auditoría antes de programar el
generador de datos, porque la clasificación de cada variable indica cómo debe
generarse: las de fuga se generan a partir del resultado, la intervención
modifica la probabilidad de mora, y la ambigua necesitaba una definición
antes de poder existir.

| # | Variable | Clasificación | Justificación |
|---|---|---|---|
| 1 | `application_date` | Identificador (eje temporal) | Existe al decidir, pero no describe al solicitante: sirve para ordenar las solicitudes en el tiempo, agruparlas por semana y hacer la partición temporal. No se usa como predictor directo. |
| 2 | `branch_id` | Predictora potencialmente válida | La sucursal se conoce desde que entra la solicitud y no cambia. También se usa para calcular el baseline histórico. Variable sensible por territorio (ver nota 3). |
| 3 | `client_id` | Identificador | Existe al decidir, pero solo identifica a la persona. No entra al modelo; sirve para que un mismo cliente no quede repartido entre desarrollo y test. |
| 4 | `age` | Predictora potencialmente válida | Viene en el formulario, así que está disponible al decidir. Variable sensible (ver nota 3). |
| 5 | `monthly_income` | Predictora potencialmente válida | Es el ingreso que la persona declara al solicitar; ese valor existe al decidir. El ingreso verificado por la revisión aparece después, así que no es este. |
| 6 | `requested_amount` | Predictora potencialmente válida | Es el monto que el cliente pide en la solicitud. El monto aprobado puede terminar siendo otro, pero ese llega después. |
| 7 | `loan_term_months` | Predictora potencialmente válida | Plazo que el cliente solicita, disponible al decidir. El plazo final puede cambiar tras la revisión. |
| 8 | `existing_debt` | Predictora potencialmente válida | Deuda que el cliente tiene al momento de solicitar, según buró o declaración. Existe al decidir. |
| 9 | `employment_type` | Predictora potencialmente válida | Se declara en el formulario, disponible al decidir. |
| 10 | `months_in_job` | Predictora potencialmente válida | Se declara en la solicitud, disponible al decidir. Viene vacía cuando el empleo es informal, porque no hay nómina que la acredite. |
| 11 | `prior_late_payments` | Predictora potencialmente válida | Cuenta atrasos de préstamos anteriores a esta solicitud. Es historial pasado, así que existe al decidir. |
| 12 | `payment_history_score` | Ambigua → válida bajo definición | Depende de cuándo se calcule. Si el sistema lo recalcula con el tiempo, el valor guardado hoy no es el que existía al decidir, y usarlo sería fuga. Lo definimos como el score calculado solo con historial previo y congelado en la fecha de la solicitud; con esa definición sí existe al decidir. Queda vacío para clientes nuevos. |
| 13 | `debt_to_income` | Predictora potencialmente válida | Se calcula con dos datos disponibles al decidir (deuda entre ingreso). Es además la base de la regla actual de la cooperativa. |
| 14 | `manual_review` | Intervención | No es un dato del solicitante: es la acción que la cooperativa toma sobre el expediente, y cambia el resultado porque la revisión reduce la probabilidad de mora. No entra al modelo; se analiza aparte. |
| 15 | `approved` | Posterior al momento de predicción | Como predecimos antes de aprobar, en ese momento esta variable todavía no existe. Solo sirve para saber qué solicitudes llegaron a tener etiqueta (las aprobadas y desembolsadas). |
| 16 | `days_past_due_6m` | Fuga como predictor · origen del target · solo auditoría | Es el resultado de los primeros 6 meses: no existe al decidir, y usarla para predecir sería fuga. Pero también tiene que estar en el dataset, porque de ella sale el target. Por eso el enunciado la lista dos veces: prohibida como predictor, obligatoria como origen de la etiqueta. |
| 17 | `default_30d` | Target | Es lo que queremos predecir. Se usa como etiqueta al entrenar y para evaluar; nunca como predictor. |
| 18 | `collection_calls_6m` | Fuga → excluir | Cobranza llama cuando ya hay atraso: el dato aparece después del resultado y por causa de él. Al decidir no existe. Se conserva solo para la demostración de fuga. |
| 19 | `restructured_after_default` | Fuga → excluir | Solo puede existir si el préstamo ya cayó en incumplimiento. Ocurre después del resultado. |
| 20 | `legal_collection_status` | Fuga → excluir | La cobranza legal empieza meses después de iniciada la mora. Al decidir no existe. |
| 21 | `p_mora_sin_intervencion` | Solo para auditoría (diseño propio) | Columna que agregamos nosotros: guarda la probabilidad de mora antes de aplicar el efecto de la revisión. En la vida real no existe; aquí sirve para medir cuánto cambia la intervención a las etiquetas. Nunca entra al modelo. |

**Resultado:** 11 variables candidatas a predictoras (10 válidas + la ambigua
ya definida), 2 identificadores, 1 intervención, 1 posterior (`approved`),
**las 4 variables de fuga del enunciado** (3 que se excluyen por completo y
`days_past_due_6m`, que además es el origen del target), 1 target y 1 columna
de auditoría propia.

**Notas**

1. `days_past_due_6m` aparece en las dos listas del enunciado (mínimas y de
   fuga). No hay contradicción, porque las listas responden preguntas
   distintas: la variable debe existir en el dataset, porque de ella sale el
   target, y a la vez no puede usarse como predictor, porque es el resultado
   mismo. Estar en los datos y entrar al modelo son cosas distintas.
2. `payment_history_score` no se podía clasificar sin antes definirla. La
   decisión del grupo quedó registrada en la tabla: score congelado a la
   fecha de la solicitud.
3. `age` y `branch_id` son válidas técnicamente pero sensibles (edad y
   territorio). En la Etapa 7 revisaremos los errores del modelo por
   sucursal, y el informe final discutirá las implicaciones de usarlas.
4. En la Etapa 5 entrenaremos un modelo con las variables de fuga incluidas,
   solo sobre la partición de desarrollo, para mostrar el desempeño casi
   perfecto y falso que producen, comparado con el modelo limpio. Los
   resultados quedarán en `reports/auditoria_fuga.md`.

# Etapa 4 · Diseño de la evaluación

En esta etapa definimos cómo se evaluará el procedimiento: qué cuenta como
un caso nuevo, qué parte de los datos se reserva para la evaluación final y
qué reglas evitan fuga durante el preprocesamiento. Estas decisiones se
toman ahora, antes de entrenar cualquier modelo, y no se modifican después.

**Qué significa un caso nuevo.** Una solicitud de una semana futura,
posiblemente de un cliente que la cooperativa nunca ha evaluado. Así usará
el modelo la cooperativa: cada semana llegan solicitudes que no existían
cuando se entrenó, muchas de personas nuevas. Nuestro test reproduce ese
escenario en su versión más exigente: todas las solicitudes de test son
posteriores en el tiempo y de clientes que el modelo no vio durante el
desarrollo. En producción el modelo también evaluará clientes que regresan;
con ellos debería irle igual o mejor, porque hay más información disponible,
así que nuestra estimación es conservadora.

**La partición.** Cortamos por fecha, no al azar:

| Bloque | Semanas | Uso |
|---|---|---|
| Entrenamiento | 1–65 (~15 meses) | Ajustar modelos y preprocesamiento |
| Validación | 66–78 (~3 meses) | Comparar modelos, elegir hiperparámetros y umbral |
| Test | 79–104 (últimos ~6 meses) | Evaluación final, una sola vez, tras congelar el protocolo |

Elegimos los últimos 6 meses como test por tres razones: es el período donde
las solicitudes superan con claridad la capacidad de 120 revisiones (llegan
entre 210 y 250 por semana), que es la situación real donde el modelo
operaría; tiene suficientes semanas y suficientes moras para comparar los
métodos con estabilidad; y deja 78 semanas para el desarrollo.

**Por qué no usamos un 80/20 aleatorio.** Por dos razones que están en
nuestros propios datos. Primero, la mora sube de 10.6% en el año 1 a 12.9%
en el año 2, y el perfil de los solicitantes cambia con el tiempo. Una
división aleatoria mezclaría solicitudes de todo el período en ambos grupos,
y el modelo se evaluaría en parte sobre épocas que ya conoce. Segundo, el
37% de las filas pertenece a clientes con más de una solicitud. Con una
división aleatoria, la solicitud de 2024 de una persona podría quedar en
entrenamiento y la de 2025 en test; el modelo acertaría en test por
reconocer un perfil casi idéntico, no porque sepa generalizar. En ambos
casos las métricas saldrían infladas y no se cumplirían en producción.

**Separación de clientes.** Usamos `client_id` para garantizar que ningún
cliente quede en dos bloques: los clientes que aparecen en test se eliminan
del desarrollo, y los que aparecen en validación se eliminan del
entrenamiento. Siempre se eliminan filas del bloque más antiguo, nunca del
más reciente, para que validación y test conserven todas sus solicitudes y
sigan representando su período.

**Cómo se evita la fuga en el preprocesamiento.** Cuatro reglas:

1. La imputación, la codificación y el escalado se ajustan solo con
   entrenamiento, dentro de un Pipeline de scikit-learn, y se aplican sin
   recalcular a validación y test.
2. El baseline histórico (tasa de mora por sucursal) se calcula solo con
   entrenamiento.
3. Ninguna estadística se calcula sobre el dataset completo.
4. Las variables de fuga y la columna de auditoría no entran a ningún
   modelo, como quedó definido en la Etapa 3.

**Por qué el test sigue siendo independiente.** Quedó definido por fecha
antes de entrenar nada; no comparte clientes con el desarrollo; ninguna
estadística ni decisión del desarrollo usa sus datos; y se abrirá una sola
vez, después de congelar el protocolo en la Etapa 6. La partición no
depende del azar — solo de fechas e identificadores — así que cualquiera
puede reproducirla.

*Nota:* como se declaró en la Etapa 1, asumimos que todos los préstamos de
test ya completaron sus 6 meses de observación al momento del análisis.

In [6]:
# =====================================================================
# Etapa 4 · Partición temporal con separación de clientes
# (usa el df de la Etapa 2, que conserva semana_idx)
# =====================================================================
SEM_FIN_TRAIN = 65   # semanas 0-64  -> entrenamiento (~15 meses)
SEM_FIN_DEV   = 78   # semanas 65-77 -> validación (~3 meses); 78+ -> test

# --- 1. Cortar test por calendario ----------------------------------
test = df[df["semana_idx"] >= SEM_FIN_DEV].copy()
dev  = df[df["semana_idx"] <  SEM_FIN_DEV].copy()

# --- 2. Limpiar desarrollo: fuera los clientes que aparecen en test -
clientes_test = set(test["client_id"])
antes = len(dev)
dev = dev[~dev["client_id"].isin(clientes_test)].copy()
removidas_por_test = antes - len(dev)

# --- 3. Cortar validación dentro del desarrollo ---------------------
val   = dev[dev["semana_idx"] >= SEM_FIN_TRAIN].copy()
train = dev[dev["semana_idx"] <  SEM_FIN_TRAIN].copy()

# --- 4. Limpiar entrenamiento: fuera los clientes de validación -----
clientes_val = set(val["client_id"])
antes = len(train)
train = train[~train["client_id"].isin(clientes_val)].copy()
removidas_por_val = antes - len(train)

# --- 5. Verificación ------------------------------------------------
def resumen(nombre, d):
    a = d["approved"] == 1
    print(f"{nombre:14s} {len(d):6,} filas | "
          f"{d.application_date.min().date()} -> {d.application_date.max().date()} | "
          f"mora (aprobadas): {d.loc[a, 'default_30d'].mean():.1%}")

print("=== PARTICIÓN ===")
resumen("Entrenamiento", train)
resumen("Validación",    val)
resumen("Test",          test)

print(f"\nFilas removidas por limpieza de clientes: "
      f"{removidas_por_test:,} del desarrollo (clientes de test) + "
      f"{removidas_por_val:,} del entrenamiento (clientes de validación)")

cruces = [len(set(train.client_id) & set(val.client_id)),
          len(set(train.client_id) & set(test.client_id)),
          len(set(val.client_id)   & set(test.client_id))]
print(f"Clientes compartidos train∩val, train∩test, val∩test: {cruces}  (debe ser [0, 0, 0])")

apps_test = test.groupby("semana_idx").size()
print(f"\nTest: {test['semana_idx'].nunique()} semanas | "
      f"solicitudes/semana promedio: {apps_test.mean():.0f} (capacidad: 120) | "
      f"moras observadas: {int(test.loc[test.approved == 1, 'default_30d'].sum()):,}")

=== PARTICIÓN ===
Entrenamiento   7,847 filas | 2024-01-01 -> 2025-03-28 | mora (aprobadas): 10.9%
Validación      2,264 filas | 2025-03-31 -> 2025-06-27 | mora (aprobadas): 13.5%
Test            5,773 filas | 2025-06-30 -> 2025-12-26 | mora (aprobadas): 13.0%

Filas removidas por limpieza de clientes: 1,639 del desarrollo (clientes de test) + 534 del entrenamiento (clientes de validación)
Clientes compartidos train∩val, train∩test, val∩test: [0, 0, 0]  (debe ser [0, 0, 0])

Test: 26 semanas | solicitudes/semana promedio: 222 (capacidad: 120) | moras observadas: 703


# Etapa 5 · Baselines y modelos

Comparamos cinco métodos de priorización sobre la validación. El test sigue
cerrado: estos resultados sirven para decidir qué se congela en la Etapa 6.

**La moneda común.** Cada método produce un puntaje de riesgo por solicitud.
Cada semana se seleccionan las 120 solicitudes con mayor puntaje (la
capacidad real) y se cuenta cuántas de las moras que de verdad ocurrieron
quedaron dentro de las seleccionadas.

**Métrica principal: recall en capacidad** — moras capturadas entre moras
totales del período. Sale de la Etapa 1: el falso negativo cuesta ~30 veces
más que el falso positivo, así que lo que importa es cuántas moras reciben
revisión. Reportamos también la precisión (qué fracción de las revisiones
acierta una mora futura) y el AUC como referencia técnica.

**Por qué no usamos exactitud (accuracy).** Con 13% de mora, "predecir que
nadie caerá" acierta el 87% sin aportar nada.

**El piso real no es cero.** En validación llegan ~174 solicitudes por
semana y se revisan 120: seleccionar al azar ya captura alrededor de dos
tercios de las moras, simplemente porque la capacidad cubre mucho. Todo
método debe superar a la suerte, no al cero. En test la capacidad cubre
menos (54% de las solicitudes), así que allí las diferencias entre métodos
se ampliarán.

**Los cinco métodos:**

1. **Trivial.** "Nadie tendrá mora": no distingue solicitudes, así que las
   120 se llenan sin criterio. Se implementa como selección aleatoria con
   semilla fija.
2. **Histórico.** Puntaje = tasa de mora de la sucursal del solicitante,
   calculada solo con entrenamiento.
3. **Operativo.** La regla vigente de la cooperativa: priorizar deuda/ingreso
   alta y atrasos previos.
4. **Regresión logística** — el modelo mínimo interpretable, con las 11
   variables auditadas.
5. **Gradient boosting** — el modelo de complejidad moderada, con las mismas
   11 variables.

**Reglas de preprocesamiento** (según la Etapa 4): la imputación y la
codificación se ajustan solo con entrenamiento, dentro de un Pipeline; cada
faltante genera además una columna indicadora, porque el vacío informa
(cliente nuevo, empleo informal); los modelos solo ven las 11 variables
auditadas.

Al final se ejecuta la demostración de fuga comprometida en la Etapa 3,
usando únicamente desarrollo.

In [7]:
# =====================================================================
# Etapa 5 · Parte 1: features, preprocesamiento y función de evaluación
# =====================================================================
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

NUM = ["age", "monthly_income", "requested_amount", "loan_term_months",
       "existing_debt", "months_in_job", "prior_late_payments",
       "payment_history_score", "debt_to_income"]
CAT = ["branch_id", "employment_type"]
CAPACIDAD = 120

# Preprocesamiento: se ajusta SOLO con entrenamiento (regla de la Etapa 4).
# add_indicator=True crea una columna "estaba vacío" por variable con
# faltantes, porque el vacío informa (cliente nuevo, empleo informal).
prep = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("esc", StandardScaler())]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
])

tr = train[train["approved"] == 1]          # solo aprobadas tienen etiqueta
X_tr, y_tr = tr[NUM + CAT], tr["default_30d"].astype(int)

def recall_en_capacidad(bloque, score, cap=CAPACIDAD):
    """Cada semana selecciona las `cap` solicitudes con mayor score y cuenta
    cuántas moras reales quedaron dentro. Devuelve (capturadas, total, seleccionadas)."""
    t = bloque[["semana_idx", "default_30d"]].copy()
    t["score"] = np.asarray(score)
    t["rank"] = t.groupby("semana_idx")["score"].rank(ascending=False, method="first")
    sel = t["rank"] <= cap
    capturadas = int((t.loc[sel, "default_30d"] == 1).sum())
    total = int((t["default_30d"] == 1).sum())
    return capturadas, total, int(sel.sum())

print(f"Entrenamiento: {len(tr):,} solicitudes aprobadas ({y_tr.mean():.1%} de mora)")

Entrenamiento: 7,445 solicitudes aprobadas (10.9% de mora)


In [8]:
# =====================================================================
# Etapa 5 · Parte 2: los cinco métodos compiten en VALIDACIÓN
# (el test sigue cerrado)
# =====================================================================
rng_eval = np.random.default_rng(SEED + 1)      # solo para desempates

modelo_logit = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000))])
modelo_gb    = Pipeline([("prep", prep), ("clf", HistGradientBoostingClassifier(random_state=SEED))])
modelo_logit.fit(X_tr, y_tr)
modelo_gb.fit(X_tr, y_tr)

tasa_sucursal = tr.groupby("branch_id")["default_30d"].mean()   # solo con train

def scores_de_todos(d):
    """Devuelve {método: puntaje} para cualquier bloque, sin reentrenar nada."""
    dti_c = np.clip(d["debt_to_income"], 0, 1.2)
    return {
        "Trivial (sin señal)":      rng_eval.random(len(d)),
        "Histórico (sucursal)":     d["branch_id"].map(tasa_sucursal).to_numpy()
                                    + rng_eval.normal(0, 1e-9, len(d)),
        "Operativo (regla actual)": (2.0 * dti_c
                                     + 0.5 * np.minimum(d["prior_late_payments"], 3)).to_numpy(),
        "Regresión logística":      modelo_logit.predict_proba(d[NUM + CAT])[:, 1],
        "Gradient boosting":        modelo_gb.predict_proba(d[NUM + CAT])[:, 1],
    }

scores_val = scores_de_todos(val)
m = (val["approved"] == 1).to_numpy()
y_val = val["default_30d"].to_numpy()

print(f"=== VALIDACIÓN: {val.semana_idx.nunique()} semanas, {len(val):,} solicitudes "
      f"(~{len(val)/val.semana_idx.nunique():.0f}/sem), capacidad {CAPACIDAD} ===\n")
print(f"{'Método':27s} {'capturadas':>11s} {'recall@cap':>11s} {'precisión':>10s} {'AUC':>7s}")
for nombre, sc in scores_val.items():
    cap_, tot_, nsel = recall_en_capacidad(val, sc)
    auc = roc_auc_score(y_val[m], np.asarray(sc)[m])
    print(f"{nombre:27s} {cap_:>7d}/{tot_:<4d} {cap_/tot_:>10.1%} {cap_/nsel:>9.1%} {auc:>7.3f}")

=== VALIDACIÓN: 13 semanas, 2,264 solicitudes (~174/sem), capacidad 120 ===

Método                       capturadas  recall@cap  precisión     AUC
Trivial (sin señal)             196/289       67.8%     12.6%   0.503
Histórico (sucursal)            204/289       70.6%     13.1%   0.534
Operativo (regla actual)        220/289       76.1%     14.1%   0.584
Regresión logística             249/289       86.2%     16.0%   0.679
Gradient boosting               240/289       83.0%     15.4%   0.641


In [9]:
# =====================================================================
# Etapa 5 · Parte 3: demostración de fuga — SOLO desarrollo
# =====================================================================
FUGA_NUM = ["collection_calls_6m", "restructured_after_default"]
FUGA_CAT = ["legal_collection_status"]
# days_past_due_6m se excluye hasta de la demostración: el target se deriva
# de ella, así que su AUC sería 1.0 por definición y no demostraría nada.
# Con los "ecos" (llamadas, reestructuración, estado legal) basta.

prep_fuga = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                      ("esc", StandardScaler())]), NUM + FUGA_NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT + FUGA_CAT),
])
modelo_fuga = Pipeline([("prep", prep_fuga), ("clf", LogisticRegression(max_iter=2000))])
modelo_fuga.fit(tr[NUM + FUGA_NUM + CAT + FUGA_CAT], y_tr)

va = val[val["approved"] == 1]
auc_fuga   = roc_auc_score(va["default_30d"],
                           modelo_fuga.predict_proba(va[NUM + FUGA_NUM + CAT + FUGA_CAT])[:, 1])
auc_limpio = roc_auc_score(va["default_30d"],
                           modelo_logit.predict_proba(va[NUM + CAT])[:, 1])

print(f"AUC con variables de fuga (validación): {auc_fuga:.3f}")
print(f"AUC del modelo limpio     (validación): {auc_limpio:.3f}")
print("\nEstos dos números completan la tabla de la sección 9 de reports/auditoria_fuga.md.")

AUC con variables de fuga (validación): 0.994
AUC del modelo limpio     (validación): 0.679

Estos dos números completan la tabla de la sección 9 de reports/auditoria_fuga.md.


# Etapa 6 · Protocolo congelado

Antes de abrir el test, registramos todas las decisiones del procedimiento
en `reports/frozen_protocol.md`: variables definitivas y excluidas, reglas
de limpieza, imputación, codificación, algoritmo, hiperparámetros, métrica,
umbral, baselines, regla de priorización, capacidad operativa y supuestos
económicos.

Resumen de lo congelado:

- El modelo seleccionado es la regresión logística con las 11 variables
  auditadas, según los resultados de validación de la Etapa 5.
- La métrica principal es el recall en capacidad: moras capturadas dentro
  de las 120 solicitudes seleccionadas cada semana.
- La comparación de referencia para decidir la implementación es el
  baseline operativo (la regla actual de la cooperativa).
- Los modelos se aplican a test tal como quedaron ajustados con el
  entrenamiento, sin reentrenar.
- El test se abre una sola vez y ninguna decisión se modifica después de
  abrirlo.

El commit que sube `reports/frozen_protocol.md` al repositorio es anterior
a la ejecución de la Etapa 7. El historial de commits documenta que las
decisiones quedaron fijadas antes de conocer cualquier resultado de test.

# Etapa 7 · Evaluación en test

El bloque de test (las últimas 26 semanas) se abre en esta sección por
primera y única vez. El protocolo quedó congelado antes en
`reports/frozen_protocol.md` (commit `c4910e9`), y el historial del
repositorio documenta ese orden.

Qué se hace aquí:

1. **Comparar los cinco métodos** con la métrica congelada: moras capturadas
   dentro de las 120 solicitudes seleccionadas cada semana.
2. **Analizar los errores** del modelo seleccionado: qué moras se le
   escapan, qué tan graves son, y cómo se reparte la captura por sucursal
   y tipo de empleo.
3. **Traducir el resultado a dinero** con los supuestos económicos fijados
   en la Etapa 1, y aplicar el criterio de decisión del protocolo.

Reglas de esta sección: los modelos se aplican tal como quedaron ajustados
con el entrenamiento, sin reentrenar; las celdas se ejecutan una sola vez y
sus resultados se reportan tal como salgan; y ninguna decisión del
procedimiento se modifica a partir de este punto.

Contexto para leer los resultados: en test llegan ~222 solicitudes por
semana y la capacidad de revisión cubre solo el 54% (en validación cubría
el 69%). Por eso los valores de recall de todos los métodos serán menores
que en validación: el período es más exigente, no los métodos peores. La
comparación relevante es la distancia entre métodos bajo las mismas
condiciones.

In [11]:
# =====================================================================
# Etapa 7 · Parte 1: APERTURA DEL TEST — se ejecuta una sola vez
# Los modelos se aplican tal como quedaron ajustados; no se reentrenan.
# =====================================================================
rng_t = np.random.default_rng(SEED + 1)
d = test
dti_c = np.clip(d["debt_to_income"], 0, 1.2)
scores_test = {
    "Trivial (sin señal)":      rng_t.random(len(d)),
    "Histórico (sucursal)":     d["branch_id"].map(tasa_sucursal).to_numpy()
                                + rng_t.normal(0, 1e-9, len(d)),
    "Operativo (regla actual)": (2.0 * dti_c
                                 + 0.5 * np.minimum(d["prior_late_payments"], 3)).to_numpy(),
    "Regresión logística":      modelo_logit.predict_proba(d[NUM + CAT])[:, 1],
    "Gradient boosting":        modelo_gb.predict_proba(d[NUM + CAT])[:, 1],
}
m = (d["approved"] == 1).to_numpy()
y_te = d["default_30d"].to_numpy()

print(f"=== TEST: {d.semana_idx.nunique()} semanas, {len(d):,} solicitudes "
      f"(~{len(d)/d.semana_idx.nunique():.0f}/sem) | la capacidad de {CAPACIDAD} "
      f"cubre el {CAPACIDAD*d.semana_idx.nunique()/len(d):.0%} ===\n")
print(f"{'Método':27s} {'capturadas':>11s} {'recall@cap':>11s} {'precisión':>10s} {'AUC':>7s}")
resultados_test = {}
for nombre, sc in scores_test.items():
    cap_, tot_, nsel = recall_en_capacidad(d, sc)
    auc = roc_auc_score(y_te[m], np.asarray(sc)[m])
    resultados_test[nombre] = cap_
    print(f"{nombre:27s} {cap_:>7d}/{tot_:<4d} {cap_/tot_:>10.1%} {cap_/nsel:>9.1%} {auc:>7.3f}")

=== TEST: 26 semanas, 5,773 solicitudes (~222/sem) | la capacidad de 120 cubre el 54% ===

Método                       capturadas  recall@cap  precisión     AUC
Trivial (sin señal)             381/703       54.2%     12.2%   0.499
Histórico (sucursal)            397/703       56.5%     12.7%   0.519
Operativo (regla actual)        445/703       63.3%     14.3%   0.602
Regresión logística             554/703       78.8%     17.8%   0.718
Gradient boosting               526/703       74.8%     16.9%   0.685


In [12]:
# =====================================================================
# Etapa 7 · Parte 2: análisis de errores de la regresión logística
# =====================================================================
t = test.copy()
t["score"] = scores_test["Regresión logística"]
t["rank"] = t.groupby("semana_idx")["score"].rank(ascending=False, method="first")
t["seleccionada"] = t["rank"] <= CAPACIDAD
mora = t["default_30d"] == 1

capt = t[mora & t.seleccionada]      # moras que sí recibieron revisión
fn   = t[mora & ~t.seleccionada]     # falsos negativos: moras sin revisión
print(f"Moras en test: {int(mora.sum())} | capturadas: {len(capt)} | sin capturar: {len(fn)}\n")

print("Perfil de las moras capturadas vs las no capturadas:")
print(f"  Severidad (días máx. de atraso): {capt.days_past_due_6m.mean():.0f} vs {fn.days_past_due_6m.mean():.0f}")
print(f"  Deuda/ingreso promedio:          {capt.debt_to_income.mean():.2f} vs {fn.debt_to_income.mean():.2f}")
print(f"  Sin historial (score vacío):     {capt.payment_history_score.isna().mean():.0%} vs {fn.payment_history_score.isna().mean():.0%}\n")

print("Tasa de captura por sucursal (compromiso de equidad de la Etapa 3):")
for b, r in t[mora].groupby("branch_id")["seleccionada"].agg(["mean", "sum", "count"]).iterrows():
    print(f"  {b:15s} {r['mean']:.0%}  ({int(r['sum'])}/{int(r['count'])})")

print("\nTasa de captura por tipo de empleo:")
for e, r in t[mora].groupby("employment_type")["seleccionada"].agg(["mean", "count"]).iterrows():
    print(f"  {e:15s} {r['mean']:.0%}  (de {int(r['count'])} moras)")

Moras en test: 703 | capturadas: 554 | sin capturar: 149

Perfil de las moras capturadas vs las no capturadas:
  Severidad (días máx. de atraso): 59 vs 60
  Deuda/ingreso promedio:          0.41 vs 0.29
  Sin historial (score vacío):     43% vs 34%

Tasa de captura por sucursal (compromiso de equidad de la Etapa 3):
  Azua            85%  (99/117)
  Baní            70%  (72/103)
  San Cristóbal   76%  (123/162)
  Santo Domingo   81%  (260/321)

Tasa de captura por tipo de empleo:
  formal          62%  (de 220 moras)
  independiente   72%  (de 138 moras)
  informal        95%  (de 325 moras)
  pensionado      35%  (de 20 moras)


In [13]:
# =====================================================================
# Etapa 7 · Parte 3: capacidad, dinero y decisión
# (supuestos económicos congelados: sección 13 del protocolo)
# =====================================================================
VALOR_CAPTURA = 0.35 * 30_000        # RD$ por mora adicional capturada
sem = test["semana_idx"].nunique()

modelo_c = resultados_test["Regresión logística"]
regla_c  = resultados_test["Operativo (regla actual)"]
delta    = modelo_c - regla_c

print(f"Con las mismas {CAPACIDAD} revisiones semanales durante {sem} semanas:")
print(f"  Regla actual : {regla_c} moras capturadas ({regla_c/sem:.1f}/semana)")
print(f"  Modelo       : {modelo_c} moras capturadas ({modelo_c/sem:.1f}/semana)")
print(f"  Diferencia   : +{delta} moras ({delta/sem:.1f}/semana)\n")

print(f"Ambos métodos gastan exactamente las mismas {CAPACIDAD*sem:,} revisiones,")
print("así que el costo de revisar es idéntico y la diferencia es beneficio neto.")
print(f"Pérdida evitada esperada adicional: {delta} × 0.35 × RD$30,000 "
      f"= RD${delta*VALOR_CAPTURA:,.0f} en el semestre "
      f"(~RD${delta*VALOR_CAPTURA/sem:,.0f}/semana)\n")

supera = modelo_c > regla_c
print(f"Criterio congelado (sección 14): ¿el modelo captura más moras que "
      f"la regla actual? -> {'SÍ' if supera else 'NO'}")
print("Decisión preliminar:", "se recomienda avanzar hacia una implementación piloto."
      if supera else "no se recomienda la implementación.")
print("La recomendación completa, con condiciones y limitaciones, se desarrolla en la Etapa 8.")

Con las mismas 120 revisiones semanales durante 26 semanas:
  Regla actual : 445 moras capturadas (17.1/semana)
  Modelo       : 554 moras capturadas (21.3/semana)
  Diferencia   : +109 moras (4.2/semana)

Ambos métodos gastan exactamente las mismas 3,120 revisiones,
así que el costo de revisar es idéntico y la diferencia es beneficio neto.
Pérdida evitada esperada adicional: 109 × 0.35 × RD$30,000 = RD$1,144,500 en el semestre (~RD$44,019/semana)

Criterio congelado (sección 14): ¿el modelo captura más moras que la regla actual? -> SÍ
Decisión preliminar: se recomienda avanzar hacia una implementación piloto.
La recomendación completa, con condiciones y limitaciones, se desarrolla en la Etapa 8.


## Lectura de los resultados de test

**Comparación.** La regresión logística capturó 554 de las 703 moras del
período (78.8%), frente a 445 (63.3%) de la regla actual: 109 moras
adicionales con las mismas 3,120 revisiones. La ventaja sobre la regla se
amplió respecto a validación (de +10 a +15 puntos), como corresponde a un
período donde la capacidad cubre solo el 54% de las solicitudes: cuando
las revisiones escasean, elegir bien vale más. El orden de los cinco
métodos se mantuvo fuera de muestra, y la logística volvió a superar al
gradient boosting (78.8% vs 74.8%), confirmando la elección congelada.

**Errores.** Las 149 moras no capturadas tienen un perfil identificable:
deuda/ingreso baja (0.29 vs 0.41 en las capturadas), más empleo formal y
más historial limpio — casos que las variables disponibles no anunciaban.
Su severidad promedio (60 días de atraso máximo) es igual a la de las
capturadas (59): el modelo no deja pasar los casos más graves, sino los
menos predecibles.

**Equidad.** La captura por sucursal va de 70% (Baní) a 85% (Azua); la
brecha queda como métrica de monitoreo. Por tipo de empleo, el modelo
captura el 95% de las moras de trabajadores informales — que concentran
casi la mitad de las moras del período — frente al 62% de los formales.
Es operativamente razonable (la verificación de ingresos aporta más donde
no hay nómina), pero implica que la carga de la revisión recae
desproporcionadamente sobre el segmento informal. El informe ejecutivo lo
discute.

**Capacidad y dinero.** Ambos métodos gastan exactamente las mismas
revisiones, así que la diferencia es beneficio neto: 109 moras × 0.35 ×
RD\$30,000 ≈ RD\$1,144,500 de pérdida evitada esperada en el semestre,
sin presupuesto adicional.

**Decisión.** El criterio congelado se cumple: el modelo supera a la regla
actual. Se recomienda avanzar hacia una implementación piloto; las
condiciones y limitaciones se desarrollan en la Etapa 8.

# Etapa 8 · Recomendación

**1. Qué procedimiento se recomienda.** La regresión logística con las 11
variables auditadas, usada para ordenar la cola semanal de revisión (120
cupos). Se recomienda avanzar a una implementación piloto en paralelo con
la regla actual, no un reemplazo inmediato.

**2. Qué mejora produjo.** En el período de test capturó 554 de las 703
moras (78.8%) frente a 445 (63.3%) de la regla actual: 109 moras
adicionales con exactamente las mismas 3,120 revisiones. Con los supuestos
económicos congelados, equivale a una pérdida evitada esperada adicional de
≈ RD\$1,144,500 en el semestre, sin presupuesto adicional.

**3. Qué errores siguen siendo importantes.** 149 moras (21%) quedan sin
revisión. Su perfil es el de casos que las variables disponibles no
anuncian: deuda/ingreso baja (0.29 vs 0.41), más empleo formal, más
historial limpio. Su severidad (60 días de atraso máximo en promedio) es
igual a la de las capturadas: no se escapan los peores casos, sino los
menos predecibles. Además, capturar no es prevenir: la revisión evita
aproximadamente el 35% de las moras de los casos tratados, así que la
pérdida se reduce, no se elimina.

**4. Limitaciones de la simulación.** Los datos son sintéticos. Las
etiquetas históricas están afectadas por la política de revisión: entre los
casos revisados, la mora observada fue 12.8% cuando su riesgo sin
intervención era 20.2% (medido con la columna de auditoría) — en datos
reales esa distorsión existe y no puede medirse. Solo las solicitudes
aprobadas tienen etiqueta. La eficacia de la revisión (35%) es un parámetro
de diseño, no una medición. Y el mundo simulado es más simple y más lineal
que el real: la ventaja de la regresión logística sobre el gradient
boosting es una conclusión de esta simulación, no una regla general.

**5. Bajo qué condiciones se implementaría.** Piloto de 3 a 6 meses en
paralelo con la regla actual; monitoreo mensual de la captura total, por
sucursal y por tipo de empleo (la carga de verificación recae más sobre el
segmento informal y debe vigilarse); el modelo solo ordena la cola de
revisión — la decisión sobre cada expediente sigue siendo de los analistas
y no se usa para rechazo automático; recalibración periódica con datos
nuevos.

**6. Qué información real sería necesaria antes de usarlo.** Historial
real de solicitudes de la cooperativa con datos reconstruidos "a la fecha"
de cada solicitud; costos contables reales de la mora, la cobranza y la
hora de analista, y las normas de provisión aplicables; la eficacia real de
la revisión, medida con una comparación cuidadosa o una prueba controlada;
y la confirmación de que el score de historial puede congelarse a la fecha
de solicitud en los sistemas de la cooperativa.

El informe ejecutivo (`reports/informe_ejecutivo.pdf`) desarrolla estos
puntos en lenguaje de negocio para la gerencia de riesgo.

# Respuestas a las preguntas del enunciado

**1. ¿Qué representa exactamente cada fila?** Una solicitud de préstamo:
una persona pidiendo un monto concreto en una fecha concreta. No es un
cliente — un mismo cliente puede tener varias solicitudes. (Etapa 1)

**2. ¿Quién recibe la predicción?** El equipo de analistas de la gerencia
de riesgo, que cada semana recibe la lista de solicitudes ordenada por
riesgo. (Etapa 1)

**3. ¿Qué acción cambia como consecuencia?** Cuáles 120 expedientes de esa
semana reciben la revisión manual detallada. Nada más: el modelo no
aprueba ni rechaza. (Etapa 1)

**4. ¿Qué variables estarán disponibles en producción?** Las 11 auditadas:
edad, ingreso declarado, monto y plazo solicitados, deuda vigente, tipo de
empleo, antigüedad laboral, atrasos previos, score de historial congelado
a la fecha, deuda/ingreso, y sucursal. (Etapa 3)

**5. ¿Qué variables contienen información futura?** Las cuatro de fuga
(`days_past_due_6m`, `collection_calls_6m`, `restructured_after_default`,
`legal_collection_status`), más `approved` (posterior al momento elegido)
y el propio target. Ninguna entra al modelo. (Etapa 3)

**6. ¿Qué intervención puede modificar el target?** La revisión manual:
en la simulación reduce la probabilidad de mora del caso revisado en ≈35%.
Por eso las etiquetas históricas reflejan el riesgo bajo la política
vigente, no el riesgo puro. (Etapas 1–3)

**7. ¿Qué significa un caso nuevo?** Una solicitud de una semana futura,
posiblemente de un cliente que la cooperativa nunca evaluó. El test
reproduce ese escenario en su versión más exigente. (Etapa 4)

**8. ¿Cuál es la partición apropiada?** Temporal con separación de
clientes: entrenamiento (semanas 1–65), validación (66–78) y test (79–104),
sin ningún `client_id` compartido entre bloques. Un 80/20 aleatorio queda
descartado por los propios datos: la mora sube de 10.6% a 12.9% entre años
y el 37% de las filas pertenece a clientes repetidos. (Etapa 4)

**9. ¿Cuál es el baseline operativo?** La regla vigente de la cooperativa:
priorizar las solicitudes con mayor deuda/ingreso y atrasos previos
(2.0 × deuda/ingreso + 0.5 × atrasos), limitada a las 120 revisiones
semanales. (Etapas 2 y 5)

**10. ¿Qué error tiene mayor costo?** El falso negativo: una mora que no
se revisa cuesta ≈RD\$30,000 esperados, contra RD\$1,000 de una revisión
gastada en un buen pagador — unas 30 veces más. Por eso la métrica
principal es cuántas moras quedan dentro de la capacidad. (Etapa 1)

**11. ¿Qué elementos se congelaron?** Todo lo que decide el resultado:
variables definitivas y excluidas, limpieza, imputación, codificación,
algoritmo e hiperparámetros, métrica, umbral por capacidad, baselines con
sus valores, regla de priorización, capacidad y supuestos económicos —
registrados en `reports/frozen_protocol.md` con commit anterior a la
apertura del test. (Etapa 6)

**12. ¿Por qué el test sigue siendo independiente?** Quedó definido por
fecha antes de entrenar nada; no comparte clientes con el desarrollo
(verificación: [0, 0, 0]); ninguna estadística ni decisión del desarrollo
usó sus datos; y se abrió una sola vez, después de congelar el protocolo.
(Etapas 4 y 6)

**13. ¿El modelo supera la práctica actual?** Sí. En test capturó 554 de
703 moras (78.8%) contra 445 (63.3%) de la regla actual: 109 moras
adicionales con exactamente las mismas 3,120 revisiones. (Etapa 7)

**14. ¿La mejora justifica la complejidad?** En dos niveles. Modelo simple
vs complejo: no — el gradient boosting quedó por debajo de la logística en
validación (83.0% vs 86.2%) y en test (74.8% vs 78.8%), así que se congeló
el modelo simple e interpretable. Modelo vs regla actual: sí — la mejora
equivale a ≈RD\$1,144,500 semestrales sin presupuesto adicional, contra el
costo de construir y mantener un modelo sencillo. (Etapas 5–7)

**15. ¿Qué aspectos del contexto real no representa la simulación?** Los
datos son sintéticos y más simples y lineales que la realidad; las
etiquetas están condicionadas por la política histórica de revisión (con
datos reales esa distorsión no puede medirse); solo las solicitudes
aprobadas tienen etiqueta; la eficacia de la revisión (35%) es un supuesto
y no una medición; y quedan fuera comportamientos reales como el fraude,
los choques macroeconómicos o los cambios regulatorios. (Etapa 8)

## Resultados principales

Evaluación en test (26 semanas, 703 moras, capacidad de 120 revisiones
por semana):

| Método | Moras capturadas | Recall@120 |
|---|---|---|
| Regla actual de la cooperativa | 445/703 | 63.3% |
| **Modelo recomendado (regresión logística)** | **554/703** | **78.8%** |

La diferencia — 109 moras adicionales con las mismas revisiones — equivale
a ≈RD\$1,144,500 de pérdida evitada esperada por semestre. Recomendación:
implementación piloto en paralelo con la regla actual. Detalles en el
notebook (Etapas 7 y 8) y en `reports/informe_ejecutivo.pdf`.